In [31]:
from elasticsearch import Elasticsearch, helpers
import json 
bulk = helpers.bulk # Insertion de masse 

es = Elasticsearch("http://elasticsearch:9200")  # connexion au server es depuis Jupyter 

In [32]:
# Suite du cours 
query = {
  "size": 0,
  "aggs": {
    "par_piece": {
      "terms": {
        "field": "play_name.keyword"
      }
    }
  }
}

res = es.search(index="shakespeare", body=query)

res["aggregations"]

{'par_piece': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'Hamlet', 'doc_count': 8},
   {'key': 'Macbeth', 'doc_count': 7},
   {'key': 'Othello', 'doc_count': 6},
   {'key': 'Romeo and Juliet', 'doc_count': 3},
   {'key': 'King Lear', 'doc_count': 2},
   {'key': 'Julius Caesar', 'doc_count': 1}]}}

### Compter le nombre de documents par speaker

In [65]:
# Pas de fonction count avec Elastic 
query = {
  "aggs": {
    "per_speaker": {
      "terms": {
        "field": "speaker.keyword"
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)

res["aggregations"]

{'per_speaker': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'HAMLET', 'doc_count': 8},
   {'key': 'MACBETH', 'doc_count': 4},
   {'key': 'OTHELLO', 'doc_count': 4},
   {'key': 'LADY MACBETH', 'doc_count': 3},
   {'key': 'ROMEO', 'doc_count': 3},
   {'key': 'IAGO', 'doc_count': 2},
   {'key': 'KING LEAR', 'doc_count': 2},
   {'key': 'MARK ANTONY', 'doc_count': 1}]}}

### Calculer l'année moyenne

In [34]:
query = {
   "aggs": {
        "avg_year": {
            "avg": {
                "field": "year"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

round(res["aggregations"]["avg_year"]["value"])

1602

### Trouver l'année min et max

In [66]:
query = {
   "aggs": {
        "min_year": {
            "min": {
                "field": "year"
            }
        },
        "max_year": {
            "max": {
                "field": "year"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

print(int( res["aggregations"]["min_year"]["value"] ) )
print(int( res["aggregations"]["max_year"]["value"] ) )

1595
1606


### Compter le nombre de documents par pièce entre 1600 et 1610

In [67]:
query = {
    "query": {
        "range": {
            "year": {
                "gte": 1600,
                "lte": 1610
            }
        }
    },
    "aggs": {
        "per_play": {
            "terms": {
                "field": "play_name.keyword"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

print( json.dumps( res["aggregations"], indent=2 ) )

{
  "per_play": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "Hamlet",
        "doc_count": 4
      },
      {
        "key": "Macbeth",
        "doc_count": 4
      },
      {
        "key": "Othello",
        "doc_count": 4
      },
      {
        "key": "King Lear",
        "doc_count": 2
      }
    ]
  }
}


### Calculer l'année moyenne par speaker

In [121]:
query = {
    "aggs": {
        "per_speaker": {
              "terms": {
                "field": "speaker.keyword"
        },
          "aggs": {
            "avg_year": {
                  "avg": {
                    "field": "year"
                  }
            }
          }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

# print(res["aggregations"]["per_speaker"]["buckets"])

g = ( (d["key"], int(d["avg_year"]["value"]) ) for d in res["aggregations"]["per_speaker"]["buckets"] )


#for v in g:
#    print(v)

In [122]:
g = ( (d["key"], d["doc_count"], int( d["moyenne_annee"]["value"] ) ) for d in res["aggregations"]["per_speaker"]["buckets"] )

In [123]:
#for r in g:
#    print(r)

### Regrouper les documents par période (avant 1600, 1600–1605, après 1605)

In [124]:
query = {
    "aggs": {
        "per_range": {
            "range": {
                "field": "year",
                "ranges": [
                    {"to": 1600},                 # avant 1600
                    {"from": 1600, "to": 1605},   # 1600–1605
                    {"from": 1605}                # après 1605
                ]
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

In [125]:
# Remarque sur l'utilisation d'une fonction à l'intérieur d'une requête 
query = {
    "aggs": {
        "per_speaker": {
            "terms": {
                "field": "speaker.keyword"
            },
            "aggs": {
                "avg_year": {
                    "avg": {
                        "field": "year"
                    }
                },
             "avg_year_int": {
                    "bucket_script": {
                        "buckets_path": {
                            "avg": "avg_year"
                        },
                        "script": "Math.round(params.avg)"
                    }
            }
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

### Combien de répliques par pièce, et quelle est la longueur moyenne des répliques ?

In [126]:
query = {
   "aggs": {
        "per_speaker": {
            "terms": {
                "field": "speaker.keyword"
            },
            "aggs": {
                "avg_reply_length_chars": {
                    "avg": {
                        "script": {
                            "source": "params['_source']['text_entry'].length()"
                        }
                    }
                },
                "avg_year_int": {
                    "bucket_script": {
                        "buckets_path": {
                            "avg": "avg_reply_length_chars"
                        },
                        "script": "Math.round(params.avg)"
                    }
                }
            }
        }
    }
}


res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

### Top 5 des speakers avec le plus de répliques

In [127]:
query = {
     "aggs": {
        "top_speakers": {
            "terms": {
                "field": "speaker.keyword",
                "size": 5,
                "order": {
                    "_count": "desc"
                }
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

#print( json.dumps( res["aggregations"], indent=2 ) )

### Nombre de speakers différents par pièce

In [128]:
query = {
    "aggs": {
        "per_playname": {
            "terms": {
                "field": "play_name.keyword"
            },
            "aggs": {
                "speakers": {
                    "terms": {
                        "field": "speaker.keyword"
                    }
                }
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)

# print( json.dumps( res["aggregations"], indent=2 ) )

### Longueur moyenne des répliques par speaker

In [129]:
query = {
    "aggs": {
        "per_speaker": {
            "terms": {
                "field": "speaker.keyword"
            },
             "aggs": {
                "avg_length": {
                    "avg": {
                        "script": {
                            "source": "params['_source']['text_entry'].length()"
                        }
                    }
                }
            }
        }
    }
}
res = es.search(index="shakespeare", size=0, body=query)

# print( json.dumps( res["aggregations"], indent=2 ) )

### Pièce avec la réplique la plus longue

In [130]:
query = {
    "aggs": {
        "per_play_name": {
            "terms": {
                "field": "play_name.keyword",
                "size": 1,
                "order": {
                    "max_length": "desc"
                }
            },
            "aggs": {
                "max_length": {
                    "max": {
                        "script": {
                            "source": "params['_source']['text_entry'].length()"
                        }
                    }
                }
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)
#print( json.dumps( res["aggregations"], indent=2 ) )

### Moyenne du nombre de répliques par pièce

In [131]:
query = {
    "aggs": {
        "per_play": {
            "terms": {
                "field": "play_name.keyword"
            }
        },
        "avg_replies": {
            "avg_bucket": {
                "buckets_path": "per_play._length"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)
#print( json.dumps( res["aggregations"], indent=2 ) )

### Répliques par année puis par pièce

In [132]:
query = {
  "aggs": {
    "par_annee": {
      "terms": {
        "field": "year"
      },
      "aggs": {
        "par_piece": {
          "terms": {
            "field": "play_name.keyword"
          }
        }
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)
#print( json.dumps( res["aggregations"], indent=2 ) )

### Pièce avec la réplique la plus longue

In [134]:
# Attention on ne peut pas faire de doc sur une keyword text_entry est un text (voir le mapping )
# pas de type text quasi jamais utilisé
query ={
  "runtime_mappings": {
    "reply_length": {
      "type": "long",
      "script": {
        "source": """
          if (params._source.containsKey('text_entry') && params._source.text_entry != null) {
            emit(params._source.text_entry.length());
          }
        """
      }
    }
  },
  "aggs": {
    "top_piece": {
      "terms": {
        "field": "play_name",
        "size": 1,
        "order": {
          "max_reply_length": "desc"
        }
      },
      "aggs": {
        "max_reply_length": {
          "max": {
            "field": "reply_length"
          }
        }
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)
#print( json.dumps( res["aggregations"], indent=2 ) )

### Regrouper les répliques par longueur (court / moyen / long)

In [136]:
# Compte tenu du mapping que l'on a fait on ne peut pas travailler sur des keyword on passe donc pas _source, les données bruts.
# type définit le type du champ créer à la volé, pas de type text quasi jamais utilisé
query = {
  "runtime_mappings": {
    "len_category": {
      "type": "keyword",
      "script": {
        "source": """
          if (params._source.containsKey('text_entry') && params._source.text_entry != null) {
            int len = params._source.text_entry.length();
            if (len < 50) emit("court");
            else if (len < 100) emit("moyen");
            else emit("long");
          }
        """
      }
    }
  },
  "aggs": {
    "per_category": {
      "terms": {
        "field": "len_category"
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)
# print( json.dumps( res["aggregations"], indent=2 ) )

{
  "per_category": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "court",
        "doc_count": 27
      }
    ]
  }
}


### Regrouper les années par décennie

In [139]:
# Pensez à gérer le cas où le champ year est vide. type définit le type du champ créer à la volé
query = {
    "runtime_mappings": {
        "decade": {
            "type": "long",
            "script": {
                "source": """
                    if (doc['year'].size() != 0) {
                        emit((doc['year'].value / 10) * 10);
                    }
                """
            }
        }
    },
    "aggs": {
        "par_decade": {
            "terms": {
                "field": "decade",
                "order": {
                    "_key": "asc"
                }
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)
print(json.dumps(res["aggregations"], indent=2))

{
  "par_decade": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": 1590,
        "doc_count": 4
      },
      {
        "key": 1600,
        "doc_count": 14
      }
    ]
  }
}


### Extraire la première lettre du speaker et regrouper dessus

In [141]:
query = {
  "runtime_mappings": {
    "first_speaker": {
      "type": "keyword",
      "script": {
        "source": """
          if (!doc['speaker'].empty) {
            emit(doc['speaker'].value.substring(0,1));
          }
        """
      }
    }
  },
  "aggs": {
    "per_char": {
      "terms": {
        "field": "first_speaker",
        "order": {
          "_key": "asc"
        }
      }
    }
  }
}

res = es.search(index="shakespeare", size=0, body=query)
print(json.dumps(res["aggregations"], indent=2))

{
  "per_char": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "H",
        "doc_count": 8
      },
      {
        "key": "I",
        "doc_count": 2
      },
      {
        "key": "K",
        "doc_count": 2
      },
      {
        "key": "L",
        "doc_count": 3
      },
      {
        "key": "M",
        "doc_count": 5
      },
      {
        "key": "O",
        "doc_count": 4
      },
      {
        "key": "R",
        "doc_count": 3
      }
    ]
  }
}


### Catégoriser les pièces : ancienne (<1600) / récente (>=1600)

In [143]:
query = {
    "runtime_mappings": {
        "category_part": {
            "type": "keyword",
            "script": {
                "source": """
                    if (doc['year'].size() != 0) {
                        emit(doc['year'].value < 1600 ? "ancienne" : "recente");
                    }
                """
            }
        }
    },
    "aggs": {
        "per_category": {
            "terms": {
                "field": "category_part"
            }
        }
    }
}

res = es.search(index="shakespeare", size=0, body=query)
print(json.dumps(res["aggregations"], indent=2))

{
  "per_category": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "recente",
        "doc_count": 14
      },
      {
        "key": "ancienne",
        "doc_count": 4
      }
    ]
  }
}
